# 🔗 Linked List Master Guide — Sean Edition

**Mental Model:** A linked list is a **treasure hunt** — each clue tells you where the next clue is, not where any specific clue is. You always start at the first clue (head). To reach clue 5 you walk through 1→2→3→4→5. There's no map of positions — only `node.next` chains. This is why insertion at head is O(1) but lookup by index is O(n).

---

## Table of Contents

| # | Section |
|---|---|
| [1](#1) | Visual Model — Structure & Pointer Mechanics |
| [2](#2) | Creating / Setup |
| [3](#3) | API & Complexity Reference |
| [4](#4) | API Demo |
| [5](#5) | Decision Map |
| [6](#6) | Pattern 1 — Reverse a Linked List (LC 206) |
| [7](#7) | Pattern 2 — Detect Cycle (LC 141) |
| [8](#8) | Pattern 3 — Merge Two Sorted Lists (LC 21) |
| [9](#9) | Pattern 4 — Remove Nth From End (LC 19) |
| [10](#10) | Pattern 5 — Reorder List (LC 143) |
| [11](#11) | Cheat Sheet & Summary |

<a id='1'></a>

## 1. 🧠 Visual Model — Structure & Pointer Mechanics

### Singly Linked List
```
head
 │
 ▼
┌───┬────┐   ┌───┬────┐   ┌───┬────┐   ┌───┬──────┐
│ 1 │ ●──┼──▶│ 2 │ ●──┼──▶│ 3 │ ●──┼──▶│ 4 │ None │
└───┴────┘   └───┴────┘   └───┴────┘   └───┴──────┘
 val  next    val  next    val  next    val  next
                                         ▲
                                        tail
```

### Doubly Linked List
```
      head
       │
       ▼
┌──────┬───┬──────┐   ┌──────┬───┬──────┐   ┌──────┬───┬──────┐
│ None │ 1 │  ●───┼──▶│  ●   │ 2 │  ●───┼──▶│  ●   │ 3 │ None │
└──────┴───┴──────┘   └──┬───┴───┴──────┘   └──┬───┴───┴──────┘
  prev  val  next        │ prev                  │ prev
                         ▼                       ▼
                    points back             points back
```

---

### Pointer Manipulation: Insert at Head — O(1)
```
BEFORE:
  head ──▶ [1] ──▶ [2] ──▶ [3] ──▶ None

STEP 1: new_node.next = head     # new node points to old head
  new ──▶ [0] ──▶ [1] ──▶ [2] ──▶ [3] ──▶ None
  head ──────────────▲  (still points to [1])

STEP 2: head = new_node          # head now points to new node
  head
   │
   ▼
  [0] ──▶ [1] ──▶ [2] ──▶ [3] ──▶ None

AFTER: Done. Two pointer assignments. O(1).
```

---

### Why O(n) for Random Access
```
Want node at index 3?

  head ──▶ [A] ──▶ [B] ──▶ [C] ──▶ [D] ──▶ None
   step 0    1       2       3       ← you are here

No shortcut. You must walk from head every time.
No pointer table. No direct jump. Just .next chains.
```

---

### Dummy Head Pattern — Eliminates Edge Cases
```
PROBLEM: Inserting at head requires special-casing.
  if head is None: head = new_node   ← annoying branch

SOLUTION: Create a dummy node before head.

  dummy ──▶ [1] ──▶ [2] ──▶ [3] ──▶ None
    │
  dummy.next is always the real head

Now every insertion is "insert after some node" — uniform logic.
Return dummy.next at the end.
```

<a id='2'></a>

## 2. 🔧 Creating / Setup

In [ ]:
from typing import Optional

# ─── Node Definition ──────────────────────────────────────────────────────────
# This is the building block. val holds data. next points to the next node.
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next  # None means "end of list"

# ─── Build a List from an Array ───────────────────────────────────────────────
# Converts [1, 2, 3, 4] → linked list 1→2→3→4→None
def make_list(vals: list) -> Optional[ListNode]:
    if not vals:
        return None
    head = ListNode(vals[0])  # first node becomes head
    curr = head
    for v in vals[1:]:        # walk through remaining values
        curr.next = ListNode(v)  # attach new node
        curr = curr.next         # advance pointer
    return head

# ─── Convert List Back to Array ───────────────────────────────────────────────
# Converts linked list 1→2→3→4→None → [1, 2, 3, 4]
# Used in test harnesses to compare results easily.
def to_list(head: Optional[ListNode]) -> list:
    result = []
    curr = head
    while curr:            # keep going until we hit None
        result.append(curr.val)
        curr = curr.next   # advance
    return result

# ─── Print List Visually ──────────────────────────────────────────────────────
# Prints: [1] → [2] → [3] → None
def print_list(head: Optional[ListNode], label: str = "") -> None:
    parts = []
    curr = head
    while curr:
        parts.append(f"[{curr.val}]")
        curr = curr.next
    arrow_str = " → ".join(parts) + " → None" if parts else "None"
    prefix = f"{label}: " if label else ""
    print(f"{prefix}{arrow_str}")

# ─── Quick Smoke Test ─────────────────────────────────────────────────────────
head = make_list([1, 2, 3, 4, 5])
print_list(head, "built list")
print("as array:", to_list(head))

# Single node
single = ListNode(42)
print_list(single, "single node")

# Empty list
empty = make_list([])
print_list(empty, "empty list")

print("ListNode, make_list, to_list, print_list defined.")

<a id='3'></a>

## 3. 📊 API & Complexity Reference

```
OPERATION                    COMPLEXITY   NOTES
──────────────────────────────────────────────────────────
Insert at head               O(1)         new_node.next = head; head = new_node
Insert at tail               O(n)         traverse to end first
Insert after known node      O(1)         given the node reference
Delete head                  O(1)         head = head.next
Delete by value              O(n)         scan to find predecessor
Delete by reference          O(1)         need predecessor node
Access by index              O(n)         no random access
Search by value              O(n)         linear scan
Reverse entire list          O(n)         three-pointer walk
Find middle                  O(n)         slow/fast pointer
Detect cycle                 O(n)         Floyd's algorithm
──────────────────────────────────────────────────────────
THINGS YOU DO NOT DO:
❌  Access by index: no ll[5] — you must traverse
❌  Delete without predecessor reference — you need prev
❌  Forget to handle empty list (head is None) — check first
❌  Forget to set last node.next = None when splitting
```

In [ ]:
# ─── Demo: make_list and to_list ──────────────────────────────────────────────
head = make_list([10, 20, 30, 40])
print_list(head, "original")

# ─── Insert at Head — O(1) ────────────────────────────────────────────────────
new_node = ListNode(5)      # create the new node
new_node.next = head        # step 1: point new node to current head
head = new_node             # step 2: update head to new node
print_list(head, "after insert 5 at head")

# ─── Insert After a Known Node — O(1) ────────────────────────────────────────
# Insert 15 after the node with value 10
curr = head
while curr and curr.val != 10:  # find node with val=10
    curr = curr.next
if curr:
    insert_node = ListNode(15)
    insert_node.next = curr.next  # step 1: new node points to curr's next
    curr.next = insert_node        # step 2: curr now points to new node
print_list(head, "after insert 15 after 10")

# ─── Delete a Node by Value — O(n) ───────────────────────────────────────────
# Delete node with value 20. Need predecessor (prev).
dummy = ListNode(0)         # dummy head avoids head deletion edge case
dummy.next = head
prev = dummy
curr = head
while curr and curr.val != 20:  # walk until we find 20
    prev = curr
    curr = curr.next
if curr:                    # found it
    prev.next = curr.next   # bypass: prev skips over curr
head = dummy.next           # real head (handles case where head was deleted)
print_list(head, "after delete 20")

# ─── Delete Head — O(1) ───────────────────────────────────────────────────────
head = head.next            # just advance head
print_list(head, "after delete head")

print("API demo complete.")

<a id='4'></a>

## 4. 🗺️ Pattern Decision Map

Read the problem signal. Pick the weapon.

```
SIGNAL IN THE PROBLEM              WHAT TO DO
──────────────────────────────────────────────────────────
"reverse a linked list"            three-pointer: prev, curr, nxt
"detect cycle"                     Floyd's fast/slow pointers
"find middle node"                 slow/fast: slow moves 1, fast moves 2
"merge two sorted lists"           dummy head + compare-and-advance
"remove nth from end"              two pointers with n-step gap
"reorder list"                     find-mid + reverse second half + interleave
"palindrome linked list"           find-mid + reverse + compare
"intersection of two lists"        count lengths, align starts
──────────────────────────────────────────────────────────
```

<a id='5'></a>

## 5. 🔄 Pattern 1 — Reverse a Linked List (LC 206)

**PROBLEM:** Given the head of a singly linked list, reverse it in place. Return the new head.

**TRICK:** Three pointers. Walk forward, flip each `.next` backward.
```
prev = None
curr = head
while curr:
    nxt = curr.next    # save next before we destroy it
    curr.next = prev   # flip the arrow
    prev = curr        # advance prev
    curr = nxt         # advance curr
return prev            # prev is the new head
```

**SLOW MOTION TRACE on [1→2→3→4→5]:**
```
Start:  prev=None  curr=1  list: 1→2→3→4→5→None

Step 1: nxt=2, flip 1.next=None, advance → prev=1, curr=2
        None←1  2→3→4→5→None

Step 2: nxt=3, flip 2.next=1,   advance → prev=2, curr=3
        None←1←2  3→4→5→None

Step 3: nxt=4, flip 3.next=2,   advance → prev=3, curr=4
        None←1←2←3  4→5→None

Step 4: nxt=5, flip 4.next=3,   advance → prev=4, curr=5
        None←1←2←3←4  5→None

Step 5: nxt=None, flip 5.next=4, advance → prev=5, curr=None
        None←1←2←3←4←5   (curr is None, loop ends)

Return prev=5 (new head)
Result: 5→4→3→2→1→None
```

**KEY INSIGHT:** Save `nxt` before flipping — once you flip `curr.next`, you lose the forward chain. Without `nxt = curr.next` first, you can't advance.

**TIME:** O(n) — one pass through the list  
**SPACE:** O(1) — three pointers only, no extra storage

In [ ]:
from typing import Optional

# ─── Node and Helpers (self-contained so this cell runs standalone) ────────────
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

def make_list(vals):
    if not vals: return None
    head = ListNode(vals[0])
    curr = head
    for v in vals[1:]:
        curr.next = ListNode(v)
        curr = curr.next
    return head

def to_list(head):
    result = []
    while head:
        result.append(head.val)
        head = head.next
    return result

def print_list(head, label=""):
    parts = []
    curr = head
    while curr:
        parts.append(f"[{curr.val}]")
        curr = curr.next
    s = " → ".join(parts) + " → None" if parts else "None"
    print(f"{label+': ' if label else ''}{s}")

# ─── LC 206: Reverse Linked List ──────────────────────────────────────────────
def reverse_list(head: Optional[ListNode]) -> Optional[ListNode]:
    """
    LC 206 — Reverse Linked List
    Approach: three-pointer walk, flip each .next backward.
    Time:  O(n) — one pass
    Space: O(1) — three pointers only
    """
    prev = None   # will become the new tail (points to None)
    curr = head   # starts at head, walks forward

    while curr:
        # SAVE next BEFORE flipping — we lose it after the flip
        nxt = curr.next      # bookmark where we're going next

        # FLIP: point current node backward instead of forward
        curr.next = prev     # reverse the arrow

        # ADVANCE both pointers one step forward
        prev = curr          # prev catches up to curr
        curr = nxt           # curr moves to saved next

    # curr is None (loop ended), prev is the last node we processed = new head
    return prev

# ─── Slow Motion Debug Trace ──────────────────────────────────────────────────
def reverse_list_debug(head: Optional[ListNode]) -> Optional[ListNode]:
    """Same algorithm with step-by-step prints."""
    prev = None
    curr = head
    step = 0
    while curr:
        step += 1
        nxt = curr.next
        curr.next = prev
        print(f"  step {step}: flipped {curr.val}.next = {prev.val if prev else None}  "
              f"→ prev={curr.val}, curr={nxt.val if nxt else None}")
        prev = curr
        curr = nxt
    return prev

# ─── Test Harness ──────────────────────────────────────────────────────────────
def test_harness(fn, test_cases):
    """Accepts a callable fn(head) -> head. Runs each test case."""
    passed = 0
    for i, (inp, expected) in enumerate(test_cases):
        head = make_list(inp)
        result = fn(head)
        got = to_list(result)
        status = "PASS" if got == expected else "FAIL"
        if status == "PASS":
            passed += 1
        print(f"  [{status}] test {i+1}: input={inp} → got={got}, expected={expected}")
    print(f"  {passed}/{len(test_cases)} passed\n")

# ─── Run Tests ────────────────────────────────────────────────────────────────
print("=== reverse_list tests ===")
test_harness(reverse_list, [
    ([1, 2, 3, 4, 5], [5, 4, 3, 2, 1]),  # standard case
    ([1, 2],          [2, 1]),            # two nodes
    ([1],             [1]),               # single node
    ([],              []),                # empty list
])

# ─── Debug Trace ──────────────────────────────────────────────────────────────
print("=== debug trace on [1,2,3,4,5] ===")
head = make_list([1, 2, 3, 4, 5])
print_list(head, "before")
reversed_head = reverse_list_debug(make_list([1, 2, 3, 4, 5]))
print_list(reversed_head, "after")

print("reverse_list defined.")

<a id='6'></a>

## 6. 🐢🐇 Pattern 2 — Detect Cycle (LC 141)

**PROBLEM:** Given the head of a linked list, determine if it contains a cycle. Return True/False.

**TRICK:** Floyd's Tortoise and Hare. Two pointers, different speeds.
```
slow = head          # moves 1 step at a time
fast = head          # moves 2 steps at a time

while fast and fast.next:
    slow = slow.next
    fast = fast.next.next
    if slow == fast:     # same node object — they met inside the cycle
        return True

return False             # fast hit None — no cycle, list terminates
```

**SLOW MOTION TRACE — cycle at position 1 (tail connects back to node 1):**
```
List structure:  [0] → [1] → [2] → [3] → [4]
                               ▲               │
                               └───────────────┘
                              (tail.next = node at index 1)

Start:  slow=[0]  fast=[0]

Step 1: slow=[1]  fast=[2]   (no match)
Step 2: slow=[2]  fast=[4]   (no match)
Step 3: slow=[3]  fast=[2]   (fast looped around)
Step 4: slow=[4]  fast=[4]   ← MATCH! Cycle detected.
```

**KEY INSIGHT:** Think of a circular track. Fast runner always laps slow runner — they must meet. If the list has no cycle, fast runner reaches the end (None) before any meeting.

Why `fast and fast.next`? Fast moves two steps, so both `fast` and `fast.next` must exist.

**TIME:** O(n) — fast pointer laps slow in at most n steps  
**SPACE:** O(1) — two pointers only

In [ ]:
from typing import Optional

# ─── Node (self-contained) ────────────────────────────────────────────────────
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

def make_list(vals):
    if not vals: return None
    head = ListNode(vals[0])
    curr = head
    for v in vals[1:]:
        curr.next = ListNode(v)
        curr = curr.next
    return head

# ─── Build a Cycle List for Testing ───────────────────────────────────────────
# pos = index where tail connects back to (-1 means no cycle)
# Example: make_cycle_list([1,2,3,4,5], 1) → tail connects back to node at index 1
def make_cycle_list(vals: list, pos: int) -> Optional[ListNode]:
    """
    Build a linked list where the tail's next points to the node at index pos.
    pos = -1 means no cycle (tail.next = None).
    """
    if not vals:
        return None

    # build the list and keep track of all nodes
    head = ListNode(vals[0])
    nodes = [head]          # index → node reference
    curr = head
    for v in vals[1:]:
        curr.next = ListNode(v)
        curr = curr.next
        nodes.append(curr)  # save reference

    # curr is now the tail node
    if pos != -1 and 0 <= pos < len(nodes):
        curr.next = nodes[pos]  # create the cycle: tail → node at pos

    return head

# ─── LC 141: Detect Cycle ─────────────────────────────────────────────────────
def has_cycle(head: Optional[ListNode]) -> bool:
    """
    LC 141 — Linked List Cycle
    Approach: Floyd's fast/slow pointer algorithm.
    Time:  O(n) — fast pointer laps slow within n steps
    Space: O(1) — two pointers only
    """
    slow = head   # tortoise: one step at a time
    fast = head   # hare: two steps at a time

    while fast and fast.next:    # fast needs two nodes ahead to be safe
        slow = slow.next         # tortoise moves 1
        fast = fast.next.next    # hare moves 2

        if slow is fast:         # same object in memory — met inside the cycle
            return True

    # fast hit None: list terminates, no cycle
    return False

# ─── Test Harness (cycle lists can't use to_list — infinite loop!) ─────────────
def test_harness_cycle(fn, test_cases):
    """Accepts fn(head) -> bool. test_cases = list of (head, expected_bool)."""
    passed = 0
    for i, (head, expected, label) in enumerate(test_cases):
        got = fn(head)
        status = "PASS" if got == expected else "FAIL"
        if status == "PASS":
            passed += 1
        print(f"  [{status}] test {i+1} ({label}): got={got}, expected={expected}")
    print(f"  {passed}/{len(test_cases)} passed\n")

# ─── Build Test Cases ─────────────────────────────────────────────────────────
# Each tuple: (head, expected_bool, label)
test_cases = [
    (make_cycle_list([3, 2, 0, -4], 1), True,  "cycle at index 1"),
    (make_cycle_list([1, 2],         0), True,  "cycle at index 0"),
    (make_cycle_list([1],           -1), False, "single node, no cycle"),
    (make_list([1, 2, 3, 4, 5]),        False, "5 nodes, no cycle"),
    (None,                              False, "empty list"),
]

print("=== has_cycle tests ===")
test_harness_cycle(has_cycle, test_cases)

print("has_cycle defined.")

<a id='7'></a>

## 7. 🔀 Pattern 3 — Merge Two Sorted Lists (LC 21)

**PROBLEM:** Given heads of two sorted linked lists, merge them into one sorted list. Return the head.

**TRICK:** Dummy head eliminates edge cases. Compare the two current heads. Attach the smaller one. Advance that pointer.
```
dummy = ListNode(0)    # anchor — dummy.next will be the real head
tail = dummy           # tail tracks where to attach the next node

while l1 and l2:
    if l1.val <= l2.val:
        tail.next = l1    # attach l1
        l1 = l1.next      # advance l1
    else:
        tail.next = l2    # attach l2
        l2 = l2.next      # advance l2
    tail = tail.next      # advance tail

tail.next = l1 or l2     # attach remaining (one is None)
return dummy.next
```

**SLOW MOTION TRACE on [1→2→4] and [1→3→4]:**
```
l1: [1]→[2]→[4]→None
l2: [1]→[3]→[4]→None
dummy → [ ] ← tail

Step 1: l1.val=1 == l2.val=1, take l1
        dummy → [1] ← tail    l1=[2]  l2=[1]

Step 2: l1.val=2 > l2.val=1, take l2
        dummy → [1] → [1] ← tail    l1=[2]  l2=[3]

Step 3: l1.val=2 < l2.val=3, take l1
        dummy → [1] → [1] → [2] ← tail    l1=[4]  l2=[3]

Step 4: l1.val=4 > l2.val=3, take l2
        dummy → [1] → [1] → [2] → [3] ← tail    l1=[4]  l2=[4]

Step 5: l1.val=4 == l2.val=4, take l1
        dummy → [1] → [1] → [2] → [3] → [4] ← tail    l1=None  l2=[4]

Loop ends (l1=None). tail.next = l2 = [4]
Result:  dummy → [1] → [1] → [2] → [3] → [4] → [4] → None
Return dummy.next: [1] → [1] → [2] → [3] → [4] → [4]
```

**KEY INSIGHT:** Dummy head means you never special-case an empty result or inserting the first node. `tail.next = l1 or l2` handles the remaining tail in one line — one of them is always None.

**TIME:** O(m+n) — visit every node once  
**SPACE:** O(1) — rearrange pointers in place, no new nodes created

In [ ]:
from typing import Optional

# ─── Node and Helpers (self-contained) ────────────────────────────────────────
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

def make_list(vals):
    if not vals: return None
    head = ListNode(vals[0])
    curr = head
    for v in vals[1:]:
        curr.next = ListNode(v)
        curr = curr.next
    return head

def to_list(head):
    result = []
    while head:
        result.append(head.val)
        head = head.next
    return result

# ─── LC 21: Merge Two Sorted Lists ────────────────────────────────────────────
def merge_two_lists(
    l1: Optional[ListNode],
    l2: Optional[ListNode]
) -> Optional[ListNode]:
    """
    LC 21 — Merge Two Sorted Lists
    Approach: dummy head + compare-and-advance.
    Time:  O(m+n) — visit every node in both lists
    Space: O(1)   — rewire existing pointers, no new nodes
    """
    dummy = ListNode(0)   # anchor node — avoids head edge case
    tail = dummy          # tail always points to the last attached node

    while l1 and l2:      # while both lists have nodes remaining
        if l1.val <= l2.val:
            tail.next = l1    # attach l1 node
            l1 = l1.next     # advance l1 pointer
        else:
            tail.next = l2    # attach l2 node
            l2 = l2.next     # advance l2 pointer
        tail = tail.next      # tail follows along

    # one list is exhausted — attach the rest of the other
    # exactly one of l1, l2 is not None (or both are None)
    tail.next = l1 if l1 else l2

    return dummy.next     # skip the dummy anchor node

# ─── Test Harness ──────────────────────────────────────────────────────────────
def test_harness(fn, test_cases):
    """Accepts fn(l1, l2) -> head. Runs each test case."""
    passed = 0
    for i, (a, b, expected) in enumerate(test_cases):
        l1 = make_list(a)
        l2 = make_list(b)
        result = fn(l1, l2)
        got = to_list(result)
        status = "PASS" if got == expected else "FAIL"
        if status == "PASS":
            passed += 1
        print(f"  [{status}] test {i+1}: {a} + {b} → got={got}, expected={expected}")
    print(f"  {passed}/{len(test_cases)} passed\n")

# ─── Run Tests ────────────────────────────────────────────────────────────────
print("=== merge_two_lists tests ===")
test_harness(merge_two_lists, [
    ([1, 2, 4], [1, 3, 4], [1, 1, 2, 3, 4, 4]),  # standard case
    ([],        [],         []),                   # both empty
    ([],        [0],        [0]),                  # l1 empty
    ([1],       [],         [1]),                  # l2 empty
    ([1, 3, 5], [2, 4, 6], [1, 2, 3, 4, 5, 6]),  # interleaved perfectly
    ([1, 1, 1], [1, 1, 1], [1, 1, 1, 1, 1, 1]),  # all duplicates
])

print("merge_two_lists defined.")

<a id='8'></a>

## 8. ✂️ Pattern 4 — Remove Nth From End (LC 19)

**PROBLEM:** Given the head of a linked list and integer n, remove the nth node from the end. Return head.

**TRICK:** Dummy head + two pointers with an n-step gap. When fast.next is None, slow.next is the target.
```
dummy = ListNode(0, head)   # dummy before head
slow = dummy
fast = dummy

# advance fast n steps ahead of slow
for _ in range(n):
    fast = fast.next

# move both until fast.next is None
while fast.next:
    slow = slow.next
    fast = fast.next

# slow.next is the node to delete
slow.next = slow.next.next

return dummy.next
```

**SLOW MOTION TRACE on [1→2→3→4→5], n=2 (remove 4, 2nd from end):**
```
dummy → [1] → [2] → [3] → [4] → [5] → None
slow = dummy   fast = dummy

Advance fast 2 steps:
  fast step 1: fast = [1]
  fast step 2: fast = [2]

State: slow=dummy  fast=[2]  (gap of 2 between them)

Move both until fast.next = None:
  iter 1: slow=[1]  fast=[3]   fast.next=[4] (not None)
  iter 2: slow=[2]  fast=[4]   fast.next=[5] (not None)
  iter 3: slow=[3]  fast=[5]   fast.next=None → STOP

slow=[3], slow.next=[4] ← this is the target
slow.next = slow.next.next = [5]

Result: [1] → [2] → [3] → [5] → None
```

**KEY INSIGHT:** The n-step gap means when fast reaches the last node, slow is exactly n nodes behind — its `.next` is the node to delete. Dummy head handles the edge case of removing the actual head node.

**TIME:** O(n) — one pass  
**SPACE:** O(1) — two pointers only

In [ ]:
from typing import Optional

# ─── Node and Helpers (self-contained) ────────────────────────────────────────
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

def make_list(vals):
    if not vals: return None
    head = ListNode(vals[0])
    curr = head
    for v in vals[1:]:
        curr.next = ListNode(v)
        curr = curr.next
    return head

def to_list(head):
    result = []
    while head:
        result.append(head.val)
        head = head.next
    return result

# ─── LC 19: Remove Nth Node From End ──────────────────────────────────────────
def remove_nth_from_end(head: Optional[ListNode], n: int) -> Optional[ListNode]:
    """
    LC 19 — Remove Nth Node From End of List
    Approach: dummy head + two pointers with n-step gap.
    Time:  O(L) — one pass, L = list length
    Space: O(1) — two pointers only
    """
    dummy = ListNode(0, head)  # dummy node before head — handles head deletion
    slow = dummy
    fast = dummy

    # advance fast n steps — now fast is n ahead of slow
    for _ in range(n):
        fast = fast.next

    # move both forward until fast.next is None (fast at last node)
    while fast.next:
        slow = slow.next
        fast = fast.next

    # slow.next is the nth node from the end — bypass it
    slow.next = slow.next.next

    return dummy.next  # return real head (handles case where original head was removed)

# ─── Debug Version ────────────────────────────────────────────────────────────
def remove_nth_from_end_debug(head: Optional[ListNode], n: int) -> Optional[ListNode]:
    """Same algorithm with step-by-step prints."""
    dummy = ListNode(0, head)
    slow = dummy
    fast = dummy

    for i in range(n):
        fast = fast.next
        print(f"  fast advance {i+1}: fast now at val={fast.val}")

    step = 0
    while fast.next:
        slow = slow.next
        fast = fast.next
        step += 1
        print(f"  both advance {step}: slow={slow.val}  fast={fast.val}")

    print(f"  target to remove: slow.next = {slow.next.val}")
    slow.next = slow.next.next
    return dummy.next

# ─── Test Harness ──────────────────────────────────────────────────────────────
def test_harness(fn, test_cases):
    """Accepts fn(head, n) -> head. Runs each test case."""
    passed = 0
    for i, (vals, n, expected) in enumerate(test_cases):
        head = make_list(vals)
        result = fn(head, n)
        got = to_list(result)
        status = "PASS" if got == expected else "FAIL"
        if status == "PASS":
            passed += 1
        print(f"  [{status}] test {i+1}: {vals} n={n} → got={got}, expected={expected}")
    print(f"  {passed}/{len(test_cases)} passed\n")

# ─── Run Tests ────────────────────────────────────────────────────────────────
print("=== remove_nth_from_end tests ===")
test_harness(remove_nth_from_end, [
    ([1, 2, 3, 4, 5], 2, [1, 2, 3, 5]),  # remove 4 (2nd from end)
    ([1],             1, []),             # remove only node
    ([1, 2],          1, [1]),            # remove tail
    ([1, 2],          2, [2]),            # remove head
    ([1, 2, 3],       3, [2, 3]),         # remove head of 3-node list
])

# ─── Debug Trace ──────────────────────────────────────────────────────────────
print("=== debug trace on [1,2,3,4,5] n=2 ===")
result = remove_nth_from_end_debug(make_list([1, 2, 3, 4, 5]), 2)
print("  result:", to_list(result))

print("remove_nth_from_end defined.")

<a id='9'></a>

## 9. 🔁 Pattern 5 — Reorder List (LC 143)

**PROBLEM:** Given list `L0→L1→…→Ln-1→Ln`, reorder it to `L0→Ln→L1→Ln-1→L2→Ln-2→…`. In-place, no return value.

Example: `[1→2→3→4→5]` → `[1→5→2→4→3]`

**TRICK:** Three phases in sequence.
```
Phase 1 — Find middle (slow/fast pointers):
  slow, fast = head, head
  while fast.next and fast.next.next:
      slow = slow.next
      fast = fast.next.next
  # slow is now at the middle node

Phase 2 — Reverse second half:
  second = slow.next
  slow.next = None      # cut the list in half!
  prev = None
  while second:
      nxt = second.next
      second.next = prev
      prev = second
      second = nxt
  second = prev         # prev is head of reversed second half

Phase 3 — Interleave first and reversed second:
  first = head
  while second:
      tmp1 = first.next
      tmp2 = second.next
      first.next = second     # insert second node after first
      second.next = tmp1      # connect to rest of first half
      first = tmp1            # advance first
      second = tmp2           # advance second
```

**SLOW MOTION TRACE on [1→2→3→4→5]:**
```
PHASE 1 — Find middle:
  slow=[1] fast=[1]
  iter 1: slow=[2] fast=[3]
  iter 2: slow=[3] fast=[5]  fast.next=None → STOP
  Middle = [3]

PHASE 2 — Reverse second half:
  second = [4]→[5]→None  (slow.next before cut)
  slow.next = None  →  first half: [1]→[2]→[3]→None
  Reverse [4]→[5]: yields [5]→[4]→None
  second = [5]→[4]→None

PHASE 3 — Interleave:
  first=[1]→[2]→[3]  second=[5]→[4]

  iter 1: tmp1=[2], tmp2=[4]
          [1].next=[5], [5].next=[2]
          first=[2], second=[4]
          list so far: [1]→[5]→[2]→[3]

  iter 2: tmp1=[3], tmp2=None
          [2].next=[4], [4].next=[3]
          first=[3], second=None
          list so far: [1]→[5]→[2]→[4]→[3]

  second=None → loop ends
Result: [1]→[5]→[2]→[4]→[3]→None
```

**KEY INSIGHT:** Three clean sub-problems. Each one is a pattern you already know: find-middle (slow/fast), reverse (three-pointer), interleave (two-pointer). The cut (`slow.next = None`) is critical — it separates the two halves so reverse doesn't corrupt the first half.

**TIME:** O(n) — three O(n) passes  
**SPACE:** O(1) — all in-place pointer rewiring

In [ ]:
from typing import Optional

# ─── Node and Helpers (self-contained) ────────────────────────────────────────
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

def make_list(vals):
    if not vals: return None
    head = ListNode(vals[0])
    curr = head
    for v in vals[1:]:
        curr.next = ListNode(v)
        curr = curr.next
    return head

def to_list(head):
    result = []
    while head:
        result.append(head.val)
        head = head.next
    return result

# ─── LC 143: Reorder List ─────────────────────────────────────────────────────
def reorder_list(head: Optional[ListNode]) -> None:
    """
    LC 143 — Reorder List
    Approach: find-middle + reverse second half + interleave.
    Modifies the list in place. No return value.
    Time:  O(n) — three linear passes
    Space: O(1) — pointer rewiring only
    """
    if not head or not head.next:
        return  # 0 or 1 node — nothing to reorder

    # ── PHASE 1: Find Middle ───────────────────────────────────────────────────
    # slow/fast: slow lands at middle when fast reaches end
    slow = head
    fast = head
    while fast.next and fast.next.next:
        slow = slow.next        # 1 step
        fast = fast.next.next   # 2 steps
    # slow is now at the middle node

    # ── PHASE 2: Reverse Second Half ──────────────────────────────────────────
    second = slow.next    # head of second half
    slow.next = None      # CUT: separate first half from second half

    prev = None
    while second:
        nxt = second.next     # save next
        second.next = prev    # flip arrow
        prev = second         # advance prev
        second = nxt          # advance second
    second = prev             # prev is now head of reversed second half

    # ── PHASE 3: Interleave First and Reversed Second ──────────────────────────
    first = head
    while second:             # second half is shorter or equal, drives the loop
        tmp1 = first.next     # save rest of first half
        tmp2 = second.next    # save rest of second half

        first.next = second   # insert second node right after first
        second.next = tmp1    # connect second to rest of first half

        first = tmp1          # advance first pointer
        second = tmp2         # advance second pointer

# ─── Test Harness ──────────────────────────────────────────────────────────────
def test_harness(fn, test_cases):
    """
    Accepts fn(head) -> None (modifies in place).
    Reads result via to_list after mutation.
    """
    passed = 0
    for i, (vals, expected) in enumerate(test_cases):
        head = make_list(vals)
        fn(head)              # modifies in place
        got = to_list(head)   # read the mutated list
        status = "PASS" if got == expected else "FAIL"
        if status == "PASS":
            passed += 1
        print(f"  [{status}] test {i+1}: {vals} → got={got}, expected={expected}")
    print(f"  {passed}/{len(test_cases)} passed\n")

# ─── Run Tests ────────────────────────────────────────────────────────────────
print("=== reorder_list tests ===")
test_harness(reorder_list, [
    ([1, 2, 3, 4],    [1, 4, 2, 3]),        # even length
    ([1, 2, 3, 4, 5], [1, 5, 2, 4, 3]),     # odd length
    ([1],             [1]),                  # single node
    ([1, 2],          [1, 2]),               # two nodes
    ([1, 2, 3],       [1, 3, 2]),            # three nodes
])

print("reorder_list defined.")

<a id='10'></a>

## 10. 🗺️ Full Pattern Decision Map

```
SIGNAL IN THE PROBLEM              PATTERN            TECHNIQUE
──────────────────────────────────────────────────────────────────────────
"reverse a linked list"            LC 206             three-pointer: prev, curr, nxt
"detect cycle"                     LC 141             Floyd's fast/slow pointers
"find middle node"                 subtask            slow/fast: slow=1, fast=2
"merge two sorted lists"           LC 21              dummy head + compare-and-advance
"remove nth from end"              LC 19              dummy + n-step gap pointers
"reorder list"                     LC 143             find-mid + reverse + interleave
"palindrome linked list"           LC 234             find-mid + reverse + compare
"intersection of two lists"        LC 160             measure lengths, align starts
"LRU cache"                        LC 146             doubly linked list + hash map
"flatten multilevel list"          LC 430             DFS-style pointer surgery
──────────────────────────────────────────────────────────────────────────

POINTER PATTERNS AT A GLANCE:
  1 pointer    → simple traversal, building, copying
  2 pointers   → fast/slow (cycle, middle, nth-from-end)
  3 pointers   → reversal (prev, curr, nxt)
  dummy head   → insertions / deletions with clean head handling
──────────────────────────────────────────────────────────────────────────
```

<a id='11'></a>

## 11. 📋 Cheat Sheet

### Template 1 — Three-Pointer Reverse
```python
prev, curr = None, head
while curr:
    nxt = curr.next    # SAVE before flip
    curr.next = prev   # FLIP
    prev = curr        # advance prev
    curr = nxt         # advance curr
return prev            # new head
```

---

### Template 2 — Fast/Slow: Detect Cycle
```python
slow = fast = head
while fast and fast.next:
    slow = slow.next
    fast = fast.next.next
    if slow is fast:
        return True
return False
```

---

### Template 3 — Fast/Slow: Find Middle
```python
slow = fast = head
while fast.next and fast.next.next:
    slow = slow.next
    fast = fast.next.next
# slow is at the middle
# For even-length list: slow at left-middle
```

---

### Template 4 — Dummy Head Merge
```python
dummy = ListNode(0)
tail = dummy
while l1 and l2:
    if l1.val <= l2.val:
        tail.next = l1
        l1 = l1.next
    else:
        tail.next = l2
        l2 = l2.next
    tail = tail.next
tail.next = l1 or l2
return dummy.next
```

---

### Template 5 — N-Gap Two Pointers (Nth From End)
```python
dummy = ListNode(0, head)
slow = fast = dummy
for _ in range(n):
    fast = fast.next    # open the gap
while fast.next:
    slow = slow.next
    fast = fast.next
slow.next = slow.next.next    # delete target
return dummy.next
```

---

### Template 6 — Reorder (Three-Phase)
```python
# Phase 1: find middle
slow, fast = head, head
while fast.next and fast.next.next:
    slow, fast = slow.next, fast.next.next

# Phase 2: reverse second half
second, slow.next = slow.next, None
prev = None
while second:
    second.next, prev, second = prev, second, second.next
second = prev

# Phase 3: interleave
first = head
while second:
    first.next, second.next = second, first.next
    first, second = first.next, second.next
```

---

### Common Gotchas
```
GOTCHA                          FIX
─────────────────────────────────────────────────────────────
Lost the forward pointer        Always: nxt = curr.next BEFORE curr.next = prev
Infinite loop on cycle test     fast and fast.next must both be checked
Off-by-one on find-middle       while fast.next AND fast.next.next (not just fast)
Head deleted without dummy      Add dummy = ListNode(0, head), return dummy.next
Forgot to cut list in half      slow.next = None before reversing second half
─────────────────────────────────────────────────────────────
```

## 12. Summary

```
╔══════════════════════════════════════════════════════════════╗
║              🔗 LINKED LIST — CORE PATTERNS                  ║
╠══════════════════════════════════════════════════════════════╣
║  REVERSE      three-pointer: prev←curr→nxt, flip, advance   ║
║  CYCLE        Floyd's: slow+1, fast+2, meet = cycle          ║
║  MIDDLE       slow+1, fast+2, stop when fast.next.next=None  ║
║  MERGE        dummy head: compare, attach smaller, advance   ║
║  NTH-END      n-gap: open gap, co-advance, slow.next=target  ║
║  REORDER      find-mid → reverse second → interleave         ║
╠══════════════════════════════════════════════════════════════╣
║  O(1) OPS: insert-head, insert-after-node, delete-head       ║
║  O(n) OPS: everything else (no random access)                ║
║  ALWAYS: check head is None before touching head.next        ║
║  ALWAYS: dummy head when inserting/deleting at head          ║
║  ALWAYS: save nxt before flipping curr.next                  ║
╚══════════════════════════════════════════════════════════════╝
```

*End of Linked List Master Guide — Sean Edition*